[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C42_Learning_Theory_Course/01_statistical_learning/01_讲解.ipynb)

# 01 · 统计学习理论（用数值实验把界钉死）

目标：把 **经验风险→真实风险的收敛**、**VC 维**、**Rademacher 复杂度**、**泛化界** 用纯 numpy 算成具体数字并 `assert` 验证。

路线：经验↔真实风险收敛($1/\sqrt n$) → Hoeffding 尾界 → 有限类 union bound 一致收敛 → 从定义估 Rademacher + Massart 上界 → VC 维(阈值/线性)打散实验 → 验证泛化界数值成立 → ✏️ 练习 → 📖 答案 → 🧪 真实数据(乳腺癌)胶囊。

> 信条：每条界都是可证伪的预测。我们把『理论上界 ≥ 实测』钉成 `assert`。

In [ ]:
import numpy as np
import math
rng = np.random.default_rng(0)
print('numpy', np.__version__)

## 1 · 经验风险以 $1/\sqrt n$ 收敛到真实风险（含半正态精确刻画）

固定假设的真实风险 $p$。经验风险的偏差 $|\hat R_n-p|$ 的**期望**有闭式：$\hat R_n-p$ 近似 $\mathcal N(0, p(1-p)/n)$，故 $\mathbb E|\hat R_n-p|\approx \sqrt{\tfrac{p(1-p)}{n}}\cdot\sqrt{\tfrac2\pi}$（半正态分布均值）。我们实测并对拍这个精确预测。

In [ ]:
p_true = 0.3
def mean_abs_gap(n, reps=4000):
    g = np.random.default_rng(123)
    gaps = [abs((g.random(n) < p_true).mean() - p_true) for _ in range(reps)]
    return np.mean(gaps)
print(f'{"n":>6s}  {"E|gap| 实测":>12s}  {"半正态理论":>12s}  {"比值":>6s}')
for n in [50, 200, 800, 3200]:
    emp = mean_abs_gap(n)
    theo = np.sqrt(p_true*(1-p_true)/n) * np.sqrt(2/np.pi)
    print(f'{n:>6d}  {emp:>12.5f}  {theo:>12.5f}  {emp/theo:>6.3f}')
# 在 n=3200 上断言理论与实测吻合到 5%
emp = mean_abs_gap(3200, reps=8000); theo = np.sqrt(p_true*(1-p_true)/3200)*np.sqrt(2/np.pi)
assert abs(emp/theo - 1) < 0.05, '经验风险偏差的期望应吻合半正态预测'
print('\n✅ 偏差 ~ 1/sqrt(n)，且其期望精确等于半正态均值 sqrt(p(1-p)/n)*sqrt(2/pi)')

## 2 · Hoeffding 集中：实测尾概率 $\le 2e^{-2nt^2}$

Hoeffding 说 $\Pr(|\hat R_n - p|\ge t)\le 2e^{-2nt^2}$。我们蒙特卡洛估计左边的尾概率，验证它确实被右边压住。

In [ ]:
def tail_prob(n, t, reps=20000):
    g = np.random.default_rng(7)
    emp = np.array([(g.random(n) < p_true).mean() for _ in range(reps)])
    return np.mean(np.abs(emp - p_true) >= t)
n = 200
print(f'{"t":>5s}  {"实测尾概率":>12s}  {"Hoeffding上界":>14s}  {"<=?":>4s}')
ok = True
for t in [0.05, 0.10, 0.15, 0.20]:
    lhs = tail_prob(n, t)
    rhs = 2*np.exp(-2*n*t**2)
    ok = ok and (lhs <= rhs + 1e-9)
    print(f'{t:>5.2f}  {lhs:>12.5f}  {rhs:>14.5f}  {str(lhs<=rhs+1e-9):>4s}')
assert ok, 'Hoeffding 尾界必须对所有 t 成立'
print('\n✅ 实测尾概率逐点压在 2 exp(-2 n t^2) 之下 —— Hoeffding 被钉死。')

## 3 · 有限假设类的一致收敛：union bound 的 $\ln|\mathcal H|$ 代价

构造一个有限类（$M$ 个固定方向的阈值分类器），看**最大**间隙 $\sup_h|\hat R_n(h)-R(h)|$ 如何随类大小 $M$ 增长，并验证它被 union bound 上界 $\sqrt{(\ln M + \ln(2/\delta))/(2n)}$ 压住。

In [ ]:
d = 8
def make_class(M, seed=1):
    g = np.random.default_rng(seed)
    W = g.standard_normal((M, d)); W /= np.linalg.norm(W, axis=1, keepdims=True)
    b = g.standard_normal(M) * 0.5
    return W, b   # h_j(x) = sign(W[j].x - b[j])
def labels_of(X, W, b):
    return np.sign(X @ W.T - b)            # (n, M)
# 数据分布 + 一个固定真标签函数(类外)制造非零风险
g = np.random.default_rng(2); w_star = g.standard_normal(d)
def sample(n, seed):
    gg = np.random.default_rng(seed)
    X = gg.standard_normal((n, d)); y = np.sign(X @ w_star)
    return X, y
# 用超大测试集估每个 h 的真实风险 R(h)
Xte, yte = sample(100000, 999)
def true_risks(W, b):
    pred = labels_of(Xte, W, b)            # (Nte, M)
    return np.mean(pred != yte[:, None], axis=0)   # (M,)
delta = 0.05; n = 200
print(f'{"M":>6s}  {"实测max间隙":>12s}  {"unionUB":>10s}  {"<=?":>4s}')
for M in [1, 10, 100, 1000]:
    W, b = make_class(M); Rt = true_risks(W, b)
    Xtr, ytr = sample(n, 50)
    Remp = np.mean(labels_of(Xtr, W, b) != ytr[:, None], axis=0)
    max_gap = np.max(np.abs(Remp - Rt))
    ub = np.sqrt((np.log(M) + np.log(2/delta)) / (2*n))
    print(f'{M:>6d}  {max_gap:>12.5f}  {ub:>10.5f}  {str(max_gap<=ub):>4s}')
print('\n✅ 最大间隙随 ln(M) 缓慢增长，且被 union bound 上界压住（界成立但偏松）。')

## 4 · 从定义估计 Rademacher 复杂度 + Massart 上界

经验 Rademacher 复杂度 $\hat{\mathfrak R}_S(A)=\mathbb E_\sigma\sup_{a\in A}\tfrac1n\langle\sigma,a\rangle$。对有限向量集，Massart 引理给上界 $\tfrac{\max\|a\|_2\sqrt{2\ln|A|}}{n}$。我们蒙特卡洛估 LHS 并验证 $\le$ Massart。

In [ ]:
def emp_rademacher(A, trials=4000, seed=3):
    '''A: (m, n) 每行一个向量 a。返回 E_sigma max_a <sigma,a>/n 的蒙特卡洛估计。'''
    g = np.random.default_rng(seed); m, n = A.shape; acc = 0.0
    for _ in range(trials):
        sigma = g.choice([-1.0, 1.0], size=n)
        acc += np.max(A @ sigma) / n
    return acc / trials
n, m = 40, 16
A = rng.choice([-1.0, 1.0], size=(m, n))          # m 个 ±1 向量（||a||=sqrt(n)）
R = emp_rademacher(A)
massart = max(np.linalg.norm(a) for a in A) * np.sqrt(2*np.log(m)) / n
print(f'经验 Rademacher 估计 = {R:.4f}')
print(f'Massart 上界         = {massart:.4f}')
assert R <= massart + 1e-9, 'Massart 有限类上界必须成立'
# sanity：单元素集合的 Rademacher 应为 0（E_sigma <sigma,a>/n = 0）
R1 = emp_rademacher(A[:1]); print(f'单元素集合的 Rademacher ≈ {R1:.4f} (应 ~0)')
assert abs(R1) < 0.03, '单假设无拟合噪声能力 -> Rademacher≈0'
print('✅ Rademacher 估计 <= Massart 上界；类越大复杂度越高。')

## 5 · VC 维：打散实验（阈值=1，$\mathbb R^2$ 线性=2=d）

VC 维 = 能被**打散**（实现全部 $2^k$ 标注）的最大点数。直接枚举验证：
(a) 阈值族 $1[x>t]$ 打散 1 点但打散不了 2 点 → VC=1；
(b) $\mathbb R^2$ 过原点线性分类器打散 2 点但一般打散不了 3 点 → VC=2=d。

In [ ]:
def threshold_realizable(points):
    '''阈值族 1[x>t] 在 points 上能实现的标注数。'''
    pts = sorted(points); k = len(pts)
    cands = [pts[0]-1] + [(pts[i]+pts[i+1])/2 for i in range(k-1)] + [pts[-1]+1]
    return len({tuple(int(x>t) for x in pts) for t in cands})
print('阈值族: 1点可实现', threshold_realizable([0.0]), '/2 ; 2点可实现', threshold_realizable([0.0,1.0]), '/4')
assert threshold_realizable([0.0]) == 2 and threshold_realizable([0.0,1.0]) < 4

def linear_shatters(P):
    '''过原点线性分类器 sign(w.x) 能实现 P 的全部 2^k 标注吗？'''
    k = len(P); g = np.random.default_rng(0); realized = set()
    for _ in range(20000):
        w = g.standard_normal(P.shape[1])
        realized.add(tuple((P @ w > 0).astype(int)))
    return len(realized) == 2**k
P2 = np.array([[1.0,0.2],[0.2,1.0]])
P3 = np.array([[1.0,0.0],[0.0,1.0],[-1.0,-1.0]])   # 第三点 = 前两点负和
print('R^2 过原点线性: 打散2点?', linear_shatters(P2), ' 打散这3点?', linear_shatters(P3))
assert linear_shatters(P2) == True, 'd=2 应能打散 2 点'
assert linear_shatters(P3) == False, '过原点线性一般打散不了 3 点 -> VC=2=d'
print('✅ 阈值 VC=1，R^2 过原点线性 VC=2=d —— 打散实验确认。')

## 6 · 把泛化界钉死：实测 $(1-\delta)$ 分位间隙 $\le$ union bound

界是『以概率 $1-\delta$ 成立』的陈述，对应间隙分布的 $(1-\delta)$ 分位数。重复抽样多次，取最大间隙的 95 百分位（$\delta=0.05$），与代入 $\delta=0.05$ 的理论上界比——对概率性界的正确实证。

In [ ]:
M = 200; delta = 0.05; n = 150
W, b = make_class(M, seed=11); Rt = true_risks(W, b)
max_gaps = []
for r in range(300):
    Xtr, ytr = sample(n, 1000 + r)
    Remp = np.mean(labels_of(Xtr, W, b) != ytr[:, None], axis=0)
    max_gaps.append(np.max(np.abs(Remp - Rt)))
q95 = np.quantile(max_gaps, 1 - delta)
ub  = np.sqrt((np.log(M) + np.log(2/delta)) / (2*n))
print(f'实测 95% 分位最大间隙 = {q95:.4f}')
print(f'union bound 理论上界  = {ub:.4f}')
print(f'界的松弛倍数(UB/实测) = {ub/q95:.2f}x  <- 保守是 union bound 的天性')
assert q95 <= ub, '泛化界(1-delta 分位)必须成立'
print('✅ 泛化界数值成立（且偏松）—— 这正是 Rademacher/PAC-Bayes 想收紧的差距。')

---
## ✏️ 练习 1：PAC 样本复杂度

agnostic 情形要让一致间隙 $\le\varepsilon$（置信 $1-\delta$）需 $n\ge\dfrac{\ln|\mathcal H|+\ln(2/\delta)}{2\varepsilon^2}$。
实现 `sample_complexity(H_size, eps, delta)` 返回所需最小**整数**样本数（向上取整）。

In [ ]:
import math
def sample_complexity(H_size, eps, delta):
    # TODO: 返回 ceil((ln|H| + ln(2/delta)) / (2 eps^2))
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
n1 = sample_complexity(1000, 0.1, 0.05)
assert n1 == math.ceil((math.log(1000)+math.log(2/0.05))/(2*0.1**2)), '公式不符'
n_a = sample_complexity(1000, 0.1, 0.05); n_b = sample_complexity(1000, 0.05, 0.05)
assert 3.5 < n_b/n_a < 4.5, 'eps 减半应使样本量约 4 倍 (1/eps^2)'
print(f'|H|=1000, eps=0.1, delta=0.05 -> n>={n1};  eps 减半后 n>={n_b} (≈4×)')
print('✅ 练习 1 通过')

## ✏️ 练习 2：VC 维 —— 1 维区间的打散

区间族 $1[a<x<b]$ 的 VC 维是 **2**：能打散 2 点，但打散不了 3 点（标注 (+,−,+) 不可实现，因为正集必须连续）。
实现 `interval_realizable(points)` 返回区间族在这些点上可实现的标注数。

In [ ]:
def interval_realizable(points):
    # TODO: 枚举所有 (a,b) 边界(点之间的中点 + 两端外)，含空区间(全0)，
    #       收集 tuple(int(a < x < b) for x in pts) 的不同标注，返回数量。
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
assert interval_realizable([0.0, 1.0]) == 4, '区间族应能打散 2 点'
r3 = interval_realizable([0.0, 1.0, 2.0])
assert r3 == 7, '3 点只能实现 7/8（缺 (+,-,+)）-> VC=2'
print(f'区间族: 2点可实现 {interval_realizable([0.0,1.0])}/4, 3点可实现 {r3}/8 -> VC=2')
print('✅ 练习 2 通过')

## ✏️ 练习 3：经验 Rademacher 复杂度（线性类闭式）

对线性类 $\{x\mapsto w^\top x:\|w\|_2\le B\}$，$\sup_{\|w\|\le B}\tfrac1n\sum_i\sigma_i (w^\top x_i)=\tfrac Bn\|\sum_i\sigma_i x_i\|_2$。
实现 `lin_emp_rademacher(X, B, trials)` 用这个闭式对每个 $\sigma$ 求 sup，再对 $\sigma$ 取平均。

In [ ]:
def lin_emp_rademacher(X, B=1.0, trials=4000, seed=5):
    # X: (n,d). 每次随机 sigma in {±1}^n: sup = (B/n)*||sum_i sigma_i x_i||_2
    # TODO: 蒙特卡洛平均这个 sup
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
Xq = rng.standard_normal((60, 5))
Rlin = lin_emp_rademacher(Xq, B=1.0, trials=6000)
ub = 1.0 * np.max(np.linalg.norm(Xq, axis=1)) / np.sqrt(len(Xq))   # B·max||x||/sqrt(n)
print(f'线性类经验 Rademacher ≈ {Rlin:.4f}, 上界 B·max||x||/sqrt(n) = {ub:.4f}')
assert Rlin <= ub + 1e-9 and Rlin > 0, '应 0 < R <= B max||x||/sqrt(n)'
assert abs(lin_emp_rademacher(Xq, B=2.0, trials=6000)/Rlin - 2) < 0.1, '复杂度应 ∝ B'
print('✅ 练习 3 通过：线性类复杂度 ∝ 范数 B，被 max||x||/sqrt(n) 控制（与维度无关）')

## ✏️ 练习 4：界的紧致度 —— union bound 有多松？

复用 worked 6 的设置，实现 `tightness(M, n, delta, reps)`：返回 `(实测95分位间隙, union上界, 松弛倍数=上界/实测)`。

In [ ]:
def tightness(M, n, delta=0.05, reps=200, seed0=2000):
    # TODO: make_class -> true_risks -> reps 次抽 n 样本算 max 间隙
    #       -> q=95分位; ub=sqrt((lnM+ln(2/delta))/(2n)); 返回 (q, ub, ub/q)
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
q, ub, slack = tightness(100, 200)
assert q <= ub and slack >= 1.0, '界必须成立且松弛倍数>=1'
print(f'M=100,n=200: 实测95%={q:.4f}, UB={ub:.4f}, 松弛={slack:.2f}x')
print('✅ 练习 4 通过：界成立但保守 —— 收紧它就是 Rademacher/PAC-Bayes 的使命')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1
def sample_complexity(H_size, eps, delta):
    return math.ceil((math.log(H_size) + math.log(2/delta)) / (2*eps**2))

In [ ]:
# 练习 2
def interval_realizable(points):
    pts = sorted(points); k = len(pts)
    cuts = [pts[0]-1] + [(pts[i]+pts[i+1])/2 for i in range(k-1)] + [pts[-1]+1]
    seen = {tuple(0 for _ in pts)}     # 空区间 -> 全 0
    for a in cuts:
        for b in cuts:
            if a < b:
                seen.add(tuple(int(a < x < b) for x in pts))
    return len(seen)

In [ ]:
# 练习 3
def lin_emp_rademacher(X, B=1.0, trials=4000, seed=5):
    g = np.random.default_rng(seed); n = X.shape[0]; acc = 0.0
    for _ in range(trials):
        sigma = g.choice([-1.0, 1.0], size=n)
        acc += (B/n) * np.linalg.norm(sigma @ X)
    return acc / trials

In [ ]:
# 练习 4
def tightness(M, n, delta=0.05, reps=200, seed0=2000):
    W, b = make_class(M, seed=11); Rt = true_risks(W, b)
    gaps = []
    for r in range(reps):
        Xtr, ytr = sample(n, seed0 + r)
        Remp = np.mean(labels_of(Xtr, W, b) != ytr[:, None], axis=0)
        gaps.append(np.max(np.abs(Remp - Rt)))
    q = np.quantile(gaps, 1 - delta)
    ub = np.sqrt((np.log(M) + np.log(2/delta)) / (2*n))
    return q, ub, ub/q

---
## 🧪 真实数据胶囊：乳腺癌数据集上估 Rademacher 复杂度并验证界

用真实数据（sklearn 乳腺癌；装包/联网失败则回退到该数据集的真实形状的合成数据）训练线性分类器类，估其经验 Rademacher 复杂度，并验证 Rademacher 泛化界 $R\le\hat R_n + 2\hat{\mathfrak R}+3\sqrt{\ln(2/\delta)/(2n)}$ 可计算且有效。

In [ ]:
try:
    from sklearn.datasets import load_breast_cancer
    data = load_breast_cancer()
    Xreal = data.data.astype(float); yreal = (data.target*2 - 1).astype(float)
    Xreal = (Xreal - Xreal.mean(0)) / (Xreal.std(0) + 1e-9)
    Xreal = Xreal / np.linalg.norm(Xreal, axis=1, keepdims=True)
    src = 'sklearn breast_cancer (真实)'
except Exception as e:
    g = np.random.default_rng(0)
    Xreal = g.standard_normal((569, 30)); Xreal /= np.linalg.norm(Xreal, axis=1, keepdims=True)
    yreal = np.sign(g.standard_normal(569)); src = f'回退合成(真实形状569×30): {type(e).__name__}'
n_all, d_all = Xreal.shape
print(f'数据来源: {src};  形状 {Xreal.shape};  每行已单位化 ||x||≈{np.linalg.norm(Xreal[0]):.3f}')

In [ ]:
def lin_rad(X, B, trials=2000, seed=0):
    g = np.random.default_rng(seed); n = len(X)
    return np.mean([(B/n)*np.linalg.norm(g.choice([-1.,1.],n) @ X) for _ in range(trials)])
B = 3.0; delta = 0.05
Rhat = lin_rad(Xreal, B, trials=3000)
ub_lin = B*np.max(np.linalg.norm(Xreal,axis=1))/np.sqrt(n_all)
print(f'真实数据上线性类经验 Rademacher = {Rhat:.4f}  (闭式上界={ub_lin:.4f})')
assert Rhat <= ub_lin + 1e-9
print('✅ 真实数据上 Rademacher 估计 <= 闭式上界')

**🧪 胶囊练习**：实现 `gen_bound(R_emp, Rhat, n, delta)` 返回 Rademacher 泛化界右端 $\hat R_n + 2\hat{\mathfrak R} + 3\sqrt{\ln(2/\delta)/(2n)}$。

In [ ]:
def gen_bound(R_emp, Rhat, n, delta):
    # TODO: 返回 R_emp + 2*Rhat + 3*sqrt(ln(2/delta)/(2n))
    raise NotImplementedError

In [ ]:
# 胶囊自测
def gen_bound(R_emp, Rhat, n, delta):
    return R_emp + 2*Rhat + 3*np.sqrt(np.log(2/delta)/(2*n))
bound = gen_bound(0.05, Rhat, n_all, delta)
assert bound >= 0.05, '界应 >= 经验风险'
print(f'设经验风险 0.05 -> Rademacher 泛化界右端 = {bound:.4f}')
print('✅ 胶囊练习通过：真实数据上的泛化界可计算且 >= 经验风险')

In [ ]:
# 📖 胶囊参考答案
def gen_bound(R_emp, Rhat, n, delta):
    return R_emp + 2*Rhat + 3*np.sqrt(np.log(2/delta)/(2*n))

---
### 小结
- 经验风险以 $1/\sqrt n$ 收敛到真实风险（偏差期望=半正态均值，已对拍）。
- 有限类：union bound 给一致收敛，代价 $\ln|\mathcal H|$；样本复杂度 agnostic $1/\varepsilon^2$、realizable $1/\varepsilon$。
- VC 维（打散）度量分类容量：阈值=1、$\mathbb R^d$ 线性=d、区间=2（均已枚举验证）。
- Rademacher 复杂度=拟合噪声的能力，数据依赖、更紧；线性类复杂度 ∝ 范数而非维度。
- 所有泛化界都用 $(1-\delta)$ 分位间隙对拍，**界成立但偏松**——这是模块 05 现代界要收紧的。

下一站：**模块 02 · 优化理论**——前面假设『ERM 解能找到』，现在问：梯度下降*能*找到吗、*多快*？